In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("walmart_retail.sales.bronze_sales")
silver_df = spark.table("walmart_retail.sales.silver_sales")
gold_fact_df = spark.table("walmart_retail.sales.gold_fact_sales")

print("Bronze rows:", bronze_df.count())
print("Silver rows:", silver_df.count())
print("Gold fact rows:", gold_fact_df.count())

Bronze rows: 8399
Silver rows: 8399
Gold fact rows: 8399


In [0]:
def check_row_count(df, expected_count, table_name):
    actual_count = df.count()
    passed = actual_count == expected_count

    return {
        "check": f"{table_name} row count",
        "expected": expected_count,
        "actual": actual_count,
        "status": "PASS" if passed else "FAIL"
    }


expected_rows = bronze_df.count()

row_count_results = [
    check_row_count(bronze_df, expected_rows, "Bronze"),
    check_row_count(silver_df, expected_rows, "Silver"),
    check_row_count(gold_fact_df, expected_rows, "Gold Fact")
]

display(spark.createDataFrame(row_count_results))

actual,check,expected,status
8399,Bronze row count,8399,PASS
8399,Silver row count,8399,PASS
8399,Gold Fact row count,8399,PASS


In [0]:
def check_nulls(df, columns, table_name):
    results = []

    for column in columns:
        null_count = df.filter(F.col(column).isNull()).count()

        results.append({
            "check": f"{table_name}.{column} null check",
            "actual": null_count,
            "expected": 0,
            "status": "PASS" if null_count == 0 else "FAIL"
        })

    return results


null_check_results = []

null_check_results.extend(
    check_nulls(
        bronze_df,
        ["order_id", "order_date", "sales", "profit"],
        "Bronze"
    )
)

null_check_results.extend(
    check_nulls(
        gold_fact_df,
        ["row_id", "order_id", "sales", "profit",
         "customer_key", "product_key",
         "date_key", "location_key"],
        "Gold Fact"
    )
)

display(spark.createDataFrame(null_check_results))

actual,check,expected,status
0,Bronze.order_id null check,0,PASS
0,Bronze.order_date null check,0,PASS
0,Bronze.sales null check,0,PASS
0,Bronze.profit null check,0,PASS
0,Gold Fact.row_id null check,0,PASS
0,Gold Fact.order_id null check,0,PASS
0,Gold Fact.sales null check,0,PASS
0,Gold Fact.profit null check,0,PASS
0,Gold Fact.customer_key null check,0,PASS
0,Gold Fact.product_key null check,0,PASS


In [0]:
def check_duplicates(df, key_columns, table_name):
    total_rows = df.count()

    distinct_rows = (
        df.select(*key_columns)
        .distinct()
        .count()
    )

    duplicate_count = total_rows - distinct_rows

    return {
        "check": f"{table_name} duplicate check",
        "actual": duplicate_count,
        "expected": 0,
        "status": "PASS" if duplicate_count == 0 else "FAIL"
    }


duplicate_results = [
    check_duplicates(
        bronze_df,
        ["row_id"],
        "Bronze"
    ),

    check_duplicates(
        silver_df,
        ["row_id"],
        "Silver"
    ),

    check_duplicates(
        gold_fact_df,
        ["row_id"],
        "Gold Fact"
    )
]

display(spark.createDataFrame(duplicate_results))

actual,check,expected,status
0,Bronze duplicate check,0,PASS
0,Silver duplicate check,0,PASS
0,Gold Fact duplicate check,0,PASS


In [0]:
def check_condition(df, condition, check_name):
    failed_count = df.filter(condition).count()

    return {
        "check": check_name,
        "actual": failed_count,
        "expected": 0,
        "status": "PASS" if failed_count == 0 else "FAIL"
    }


business_rule_results = [

    check_condition(
        silver_df,
        F.col("order_quantity") <= 0,
        "Silver invalid order quantity"
    ),

    check_condition(
        silver_df,
        F.col("sales") < 0,
        "Silver negative sales"
    ),

    check_condition(
        silver_df,
        F.col("discount") < 0,
        "Silver negative discount"
    ),

    check_condition(
        silver_df,
        F.col("discount") > 1,
        "Silver discount greater than 100%"
    ),

    check_condition(
        silver_df,
        F.col("ship_date") < F.col("order_date"),
        "Silver ship date before order date"
    ),

    check_condition(
        gold_fact_df,
        F.col("shipping_days") < 0,
        "Gold negative shipping days"
    )
]

display(spark.createDataFrame(business_rule_results))

actual,check,expected,status
0,Silver invalid order quantity,0,PASS
0,Silver negative sales,0,PASS
0,Silver negative discount,0,PASS
0,Silver discount greater than 100%,0,PASS
0,Silver ship date before order date,0,PASS
0,Gold negative shipping days,0,PASS


In [0]:
dq_results = []

# Row count checks
dq_results.extend(row_count_results)

# Null checks
dq_results.extend(null_check_results)

# Duplicate checks
dq_results.extend(duplicate_results)

# Business rule checks
dq_results.extend(business_rule_results)

dq_report = spark.createDataFrame(dq_results)

display(
    dq_report.orderBy(
        F.when(F.col("status") == "FAIL", 1).otherwise(2),
        "check"
    )
)

actual,check,expected,status
0,Bronze duplicate check,0,PASS
8399,Bronze row count,8399,PASS
0,Bronze.order_date null check,0,PASS
0,Bronze.order_id null check,0,PASS
0,Bronze.profit null check,0,PASS
0,Bronze.sales null check,0,PASS
0,Gold Fact duplicate check,0,PASS
8399,Gold Fact row count,8399,PASS
0,Gold Fact.customer_key null check,0,PASS
0,Gold Fact.date_key null check,0,PASS


In [0]:
total_checks = dq_report.count()

failed_checks = (
    dq_report
    .filter(F.col("status") == "FAIL")
    .count()
)

passed_checks = total_checks - failed_checks

overall_status = "PASS" if failed_checks == 0 else "FAIL"

print(f"Total checks : {total_checks}")
print(f"Passed       : {passed_checks}")
print(f"Failed       : {failed_checks}")
print(f"Overall status: {overall_status}")

Total checks : 24
Passed       : 24
Failed       : 0
Overall status: PASS


In [0]:
if overall_status == "FAIL":
    raise Exception(
        f"Data quality validation failed: "
        f"{failed_checks} of {total_checks} checks failed."
    )

print("Data quality validation completed successfully.")

Data quality validation completed successfully.
